# 03b · Localize: does the causal step see where the damage is?

**What this step does.** For seed 0, every mechanism and dose, reruns DirectLiNGAM (same prior knowledge as Stage 0) on the training variables with the **noisy** label, and records the strength of every edge **into the label**.

**Why it matters.** Under *targeted* noise in older patients, the age → label relationship is damaged, so the edge from `age_years` into the label should change most as the dose rises, while other edges stay put. Under *uniform* noise, all edges into the label should weaken together. This shows the causal step can **localize** a problem, which confident learning alone cannot.

**Honest framing for the paper:** this is supporting evidence, not the main detector. Binary variables strain LiNGAM's assumptions (the gate says BOUNDED), so we report the *pattern* across doses, not the exact coefficients.

**Inputs:** `data/{dataset}_train.csv`, `results/noisy/*__s0.csv` · **Outputs:** `results/03b_localize.csv`.

For CDC, watch `AnyHealthcare` and `NoDocbcCost` (the targeted stratum is defined by access to care).

**Which datasets run:** `li.RUN_ONLY` in `li_lib.py` (now: Taiwan credit only). Results for the other datasets are kept: the CSVs are merged, not overwritten.

In [1]:
# --- setup: run this cell first -------------------------------------------
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))          # li_lib.py sits next to the notebooks
import li_lib as li
pd.set_option("display.width", 160, "display.max_columns", 30)
print("li_lib", li.VERSION, "| project folder:", li.ROOT)
li.versions()

li_lib 1.1.0 | project folder: D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity


,package,version
0,numpy,2.2.6
1,pandas,3.0.6
2,scikit-learn,1.9.1
3,xgboost,3.4.1
4,cleanlab,2.9.0
5,lingam,1.13.0
6,causal-learn,0.1.4.8


In [2]:
rows, t0 = [], time.time()
for name, cfg in li.run_configs().items():
    cfg = li.apply_frozen(cfg)
    tr = li.load_split(cfg, "train")
    f, prot = li.stage0_frame(cfg, tr)
    for mech in li.MECHANISMS:
        for dose in li.DOSES:
            nz = pd.read_csv(li.RESULTS / "noisy" / f"{name}__{mech}__{dose:.2f}__s0.csv")
            g = f.copy(); g[cfg.label] = nz.y_noisy.to_numpy(float)
            coefs = li.lingam_edges_into_label(g, cfg.label, prot, seed=0)
            for var, b in coefs.items():
                rows.append({"dataset": name, "mechanism": mech, "dose": dose, "variable": var, "coef_into_label": b})
    print(f"{name}: done ({time.time()-t0:.0f}s)")
loc = pd.DataFrame(rows)
loc = li.save_merged(loc, "03b_localize.csv", show=0)

taiwancredit: done (50s)
kept 945 existing rows for ['cdcdiabetes', 'diabetes130', 'southgerman']
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\03b_localize.csv  (1,260 rows x 5 cols)


## Which edges move most?
Change in each edge's coefficient from dose 0 to dose 0.40. **Look for:** under targeted noise, `age_years` at or near the top.

In [3]:
piv = loc.pivot_table(index=["dataset", "mechanism", "variable"], columns="dose", values="coef_into_label")
piv["change_0_to_040"] = piv[0.40] - piv[0.00]
(piv.reset_index().assign(abs_change=lambda d: d.change_0_to_040.abs())
    .sort_values(["dataset", "mechanism", "abs_change"], ascending=[True, True, False])
    .groupby(["dataset", "mechanism"]).head(4).round(4))

dose,dataset,mechanism,variable,0.0,0.03,0.05,0.1,0.2,0.3,0.4,change_0_to_040,abs_change
7,cdcdiabetes,class_conditional,GenHlth,0.1612,0.1516,0.1537,0.1492,0.1428,0.1251,0.1179,-0.0433,0.0433
2,cdcdiabetes,class_conditional,BMI,0.1342,0.1324,0.1287,0.1262,0.1187,0.1069,0.0997,-0.0346,0.0346
9,cdcdiabetes,class_conditional,HighBP,0.1044,0.1013,0.1004,0.0961,0.0881,0.0881,0.0698,-0.0345,0.0345
0,cdcdiabetes,class_conditional,Age,0.0847,0.0825,0.0803,0.0815,0.0755,0.0611,0.0627,-0.0220,0.0220
35,cdcdiabetes,targeted,NoDocbcCost,0.0000,0.0000,0.0000,-0.0260,-0.0324,-0.0530,-0.0635,-0.0635,0.0635
25,cdcdiabetes,targeted,DiffWalk,0.0599,0.0602,0.0604,0.0533,0.0558,0.0565,0.0541,-0.0058,0.0058
28,cdcdiabetes,targeted,GenHlth,0.1612,0.1606,0.1591,0.1572,0.1613,0.1587,0.1560,-0.0051,0.0051
31,cdcdiabetes,targeted,HighChol,0.0748,0.0742,0.0742,0.0741,0.0737,0.0712,0.0704,-0.0044,0.0044
44,cdcdiabetes,uniform,BMI,0.1342,0.1211,0.1100,0.0948,0.0689,0.0489,0.0000,-0.1342,0.1342
49,cdcdiabetes,uniform,GenHlth,0.1612,0.1446,0.1351,0.1102,0.0803,0.0611,0.0438,-0.1173,0.1173
